# Preprocess Data with Rest Epochs

In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import mne
import pandas as pd

In [6]:
LOADED_ROOT = Path("../data/loaded")
SAVE_ROOT = Path("../data/preprocessed_rest_epochs")

RESULTS_ROOT = Path("../results/preprocessed_data_rest_epochs")
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

files = sorted(LOADED_ROOT.rglob("*.npz"))

print(f"Found {len(files)} files")

event_summary = []

for file in files:

    ## ---------- LOAD DATA ----------

    subject = file.parent.parent.name
    task = file.parent.name
    run = file.stem

    print(subject, task, run)
    
    data = np.load(file, allow_pickle=True)

    signals = data["signals"]
    aux = data["aux"]
    fs = int(data["fs"])
    channel_names = data["channel_names"].tolist()
    positions = data["positions"]
    orientations = data["orientations"]

    print(file.relative_to(LOADED_ROOT))
    print(signals.shape)
    print(aux.shape)
    print(fs)

    ## ---------- CREATE MNE ----------

    info = mne.create_info(
        ch_names=channel_names,
        sfreq=fs,
        ch_types=["mag"] * len(channel_names)
    )

    raw = mne.io.RawArray(
        signals,
        info
    )

    print(raw)

    ## ---------- FILTERING ----------

    raw_filt = raw.copy()

    raw_filt.filter(
        l_freq=1.0,
        h_freq=40.0
    )
    
    raw_filt.notch_filter(
        freqs=[60, 120]
    )

    ## ---------- EVENT DETECTION ----------

    task = file.parent.name.lower()
    print(task)
    print(aux.shape)

    if task != "rest":

        if task == "motor":
            trigger = aux[2]
            threshold = 0.5
        else:
            trigger = aux[0]
            threshold = 2.0      

        binary = trigger > threshold

        onsets = np.where(
            np.diff(binary.astype(int)) == 1
        )[0]

        n_detected = len(onsets)

        if task == "auditory":
            # The auditory stimulus reaches the ears 60 ms after the recorded trigger
            auditory_delay = 0.060  # seconds
            delay_samples = int(round(auditory_delay * fs))

            onsets = onsets + delay_samples

            print(
                f"Applied auditory stimulus delay: "
                f"{auditory_delay * 1000:.0f} ms "
                f"({delay_samples} samples)"
            )

        print(
            "Number of events:",
            len(onsets)
        )

        events = np.column_stack(
            [
                onsets,
                np.zeros(
                    len(onsets),
                    dtype=int
                ),
                np.ones(
                    len(onsets),
                    dtype=int
                )
            ]
        )

        print(events.shape)

        ## ---------- EPOCHING ----------

        epochs = mne.Epochs(
            raw_filt,
            events,
            event_id=1,
            tmin=-0.2,
            tmax=0.5,
            baseline=(-0.2, 0),
            preload=True
        )

        print(epochs)
        n_retained = len(epochs)
        n_dropped = n_detected - n_retained

        event_summary.append({
            "Subject": subject,
            "Task": task,
            "Run": run,
            "Detected events": n_detected,
            "Retained epochs": n_retained,
            "Dropped epochs": n_dropped
        })
        
        X = epochs.get_data()
        times = epochs.times
        print(
            "Epochs:",
            X.shape
        )

    else: # Rest Task

        print("Creating fixed-length resting-state windows...")

        # Number of time points per epoch
        N_TIMES = 1401

        rest_data = raw_filt.get_data()

        n_channels, n_samples = rest_data.shape

        n_windows = (
            n_samples - N_TIMES
        ) // N_TIMES + 1

        print("Number of rest windows:", n_windows)

        X = np.stack(
            [
                rest_data[
                    :,
                    start:start + N_TIMES
                ]
                for start in range(
                    0,
                    n_windows * N_TIMES,
                    N_TIMES
                )
            ]
        )

        print(
            "Number of Rest windows:", X.shape
        )

        times = np.arange(
            N_TIMES
        ) / fs

        n_detected = n_windows
        n_retained = n_windows
        n_dropped = 0

        event_summary.append({
            "Subject": subject,
            "Task": task,
            "Run": run,
            "Detected events": n_detected,
            "Retained epochs": n_retained,
            "Dropped epochs": n_dropped
        })

    ## ---------- TENSORS ---------- 

    print(
        "Final tensor shape:",
        X.shape
    )

    evoked = epochs.average()

    gfp = np.std(
        evoked.data,
        axis=0
    )

    SAVE_PATH = (
        SAVE_ROOT /
        subject /
        task /
        f"{run}_epochs.npz"
    )
    
    SAVE_PATH.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    np.savez_compressed(
        SAVE_PATH,
        epochs=X,
        times=times,
        fs=fs,
        positions=positions,
        orientations=orientations,
        channel_names=np.array(
            channel_names,
            dtype=object
        )
    )

    print(f"Saved: {SAVE_PATH}")
    print(X.shape)

event_df = pd.DataFrame(event_summary)

print("\n========== EVENT SUMMARY ==========")
print(event_df.to_string(index=False))

subject_task_summary = (
    event_df
    .groupby(["Subject", "Task"])
    .agg(
        Runs=("Run", "count"),
        Events=("Detected events", "sum"),
        Retained_epochs=("Retained epochs", "sum"),
        Dropped_epochs=("Dropped epochs", "sum")
    )
    .reset_index()
)

print("\n========== SUBJECT × TASK SUMMARY ==========")
print(subject_task_summary.to_string(index=False))

task_summary = (
    event_df
    .groupby("Task")
    .agg(
        Total_runs=("Run", "count"),
        Total_events=("Detected events", "sum"),
        Total_retained_epochs=("Retained epochs", "sum"),
        Total_dropped_epochs=("Dropped epochs", "sum")
    )
    .reset_index()
)

print("\n========== TASK SUMMARY ==========")
print(task_summary.to_string(index=False))

event_df.to_csv(
    RESULTS_ROOT / "event_counts_per_run.csv",
    index=False
)

subject_task_summary.to_csv(
    RESULTS_ROOT / "event_counts_subject_task.csv",
    index=False
)

task_summary.to_csv(
    RESULTS_ROOT / "event_counts_task.csv",
    index=False
)


Found 32 files
002 auditory run01
002/auditory/run01.npz
(30, 856000)
(1, 856000)
2000
Creating RawArray with float64 data, n_channels=30, n_times=856000
    Range : 0 ... 855999 =      0.000 ...   428.000 secs
Ready.
<RawArray | 30 x 856000 (428.0 s), ~195.9 MiB, data loaded>
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 6601 samples (3.300 s)

Filtering raw data in 1 contiguous segment
Setting up band-stop filter

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal